# BBEATSx Phase 3: Ablation Studies

This notebook evaluates the core design choices of BBEATSx by conducting ablation studies as outlined in the research plan §3.4:
1. **Trend Block Extrapolation**: Spline Trend vs TVP Trend vs Tree-on-t Trend (Lemma 2.3).
2. **Error Variance Model**: Homoscedastic vs Stochastic Volatility errors under clustered volatility (Prop 2.4).
3. **Asymmetric Priors**: Identifiability priors on vs off (Thm 2.2).
4. **Sum-to-zero seasonal centering**: Centering on vs off (Thm 2.2).

*(Note: `multistep="direct"` is a reserved config field, but recursive forecasting is used throughout as recommended).* 

## Google Colab Setup

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Setting up workspace...")
    import os
    if not os.path.abspath(".").endswith("simulations"):
        if not os.path.exists("BBEATSx"):
            !git clone https://github.com/hugogobato/BBEATSx.git
        !pip install ./BBEATSx
        %cd BBEATSx/simulations
    else:
        print("Already in simulations directory.")

import sys
import os
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath(".."))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bbeatsx import BBEATSx, make_config
import utils

In [ ]:
NUM_SIMULATIONS = 30  # Set to 100 for final results on Colab
LEVEL = 0.90

## 1. Trend Block Extrapolation (Tree vs Spline vs TVP)

We test Lemma 2.3 by comparing how the three trend modes extrapolate over a long horizon $H=35$ on DGP 1 (linear trend). Because tree-trend splits on time $t$, it should revert to a constant prediction out-of-sample and fail to extrapolate, leading to rising RMSE and 0% coverage on later horizons. Spline and TVP should extrapolate linearly.

In [ ]:
H_long = 35
results_trend = []
sample_trajectories = {}

print("Running Trend extrapolation ablation...")
for i in range(NUM_SIMULATIONS):
    t, y, true_comps = utils.generate_dgp1(n=200, seed=i)
    n_train = len(y) - H_long
    y_train, y_test = y[:n_train], y[n_train:]
    
    for mode in ["spline", "tvp", "tree"]:
        cfg = make_config(
            periods=[(12, 3)],
            lags=(1,),
            trend=mode,
            errors="homo",
            num_gfr=10,
            num_burnin=120,
            num_mcmc=250,
            seed=i
        )
        
        model = BBEATSx(cfg).fit(y_train)
        fc = model.forecast(H_long)
        
        # We evaluate metrics on the later part of the forecast horizon where extrapolation is key
        # (horizons 15 to 35)
        eval_start = 15
        y_test_extrap = y_test[eval_start:]
        fc_samples_extrap = fc.samples[eval_start:]
        fc_mean_extrap = fc.mean()[eval_start:]
        
        rmse = utils.compute_rmse(y_test_extrap, fc_mean_extrap)
        cov = utils.compute_coverage(y_test_extrap, fc_samples_extrap, level=LEVEL)
        
        results_trend.append({
            "trial": i,
            "mode": mode,
            "extrap_rmse": rmse,
            "extrap_cov": cov
        })
        
        if i == 0:
            sample_trajectories[mode] = {
                "mean": fc.mean(),
                "lower": np.quantile(fc.samples, (1-LEVEL)/2, axis=1),
                "upper": np.quantile(fc.samples, 1-(1-LEVEL)/2, axis=1),
                "y_test": y_test,
                "y_train": y_train
            }
print("Trend ablation completed.")

In [ ]:
df_trend = pd.DataFrame(results_trend)
print("\n--- Trend Extrapolation Ablation Summary (Horizon 15-35) ---")
print(df_trend.groupby("mode")[["extrap_rmse", "extrap_cov"]].mean())

# Plot representative trajectory to show tree trend flatlining
plt.figure(figsize=(10, 5))
t_train = np.arange(len(sample_trajectories["spline"]["y_train"]))
t_test = np.arange(len(t_train), len(t_train) + H_long)

plt.plot(t_train[-50:], sample_trajectories["spline"]["y_train"][-50:], color='black', label='History')
plt.plot(t_test, sample_trajectories["spline"]["y_test"], color='grey', linestyle='--', label='Actual Future')

colors = {"spline": "blue", "tvp": "green", "tree": "red"}
for mode in ["spline", "tvp", "tree"]:
    traj = sample_trajectories[mode]
    plt.plot(t_test, traj["mean"], color=colors[mode], label=f"{mode} (mean)")
    plt.fill_between(t_test, traj["lower"], traj["upper"], color=colors[mode], alpha=0.15)

plt.title("Trend Mode Extrapolation Comparison (Plan §3.4)")
plt.legend()
plt.show()

## 2. Error Model: Homoscedastic vs Stochastic Volatility (SV)

We validate the calibration gain from modeling heteroscedasticity using DGP 2 (which has volatility clustering). We compare `errors="homo"` vs `errors="sv"` on CRPS, pinball loss, and empirical coverage.

In [ ]:
H = 20
results_errors = []

print("Running Error model ablation on DGP 2...")
for i in range(NUM_SIMULATIONS):
    t, y, true_comps = utils.generate_dgp2(n=250, seed=i)
    n_train = len(y) - H
    y_train, y_test = y[:n_train], y[n_train:]
    
    for mode in ["homo", "sv"]:
        cfg = make_config(
            periods=[(12, 3), (7, 2)],
            lags=(1, 2),
            trend="spline",
            errors=mode,
            num_gfr=10,
            num_burnin=120,
            num_mcmc=250,
            seed=i
        )
        
        model = BBEATSx(cfg).fit(y_train)
        fc = model.forecast(H)
        
        crps = utils.compute_crps(y_test, fc.samples)
        cov = utils.compute_coverage(y_test, fc.samples, level=LEVEL)
        width = np.mean(np.quantile(fc.samples, 1-(1-LEVEL)/2, axis=1) - np.quantile(fc.samples, (1-LEVEL)/2, axis=1))
        
        results_errors.append({
            "trial": i,
            "mode": mode,
            "crps": crps,
            "coverage": cov,
            "interval_width": width
        })
print("Error model ablation completed.")

In [ ]:
df_errors = pd.DataFrame(results_errors)
print("\n--- Error Variance Ablation Summary ---")
print(df_errors.groupby("mode")[["crps", "coverage", "interval_width"]].mean())

## 3. Asymmetric Component Priors (Identifiability)

We evaluate how the asymmetric regularizing prior prevents the generic block from "stealing" variance from the trend and seasonal blocks. We compare `asymmetric=True` vs `asymmetric=False` on DGP 1, measuring in-sample component recovery RMSE.

In [ ]:
results_asym = []

print("Running Asymmetric prior ablation on DGP 1...")
for i in range(NUM_SIMULATIONS):
    t, y, true_comps = utils.generate_dgp1(n=200, seed=i)
    
    for asym in [True, False]:
        cfg = make_config(
            periods=[(12, 3)],
            lags=(1,),
            trend="spline",
            errors="homo",
            asymmetric=asym,
            num_gfr=10,
            num_burnin=120,
            num_mcmc=250,
            seed=i
        )
        
        model = BBEATSx(cfg).fit(y)
        s = model.sampler_
        mean, std = s.y_mean_, s.y_std_
        in_sample_draws = s.in_sample_components()
        
        pred_comps = {
            "trend": mean + std * in_sample_draws["trend"],
            "seasonal": std * in_sample_draws["seasonal"],
            "generic": std * in_sample_draws["generic"]
        }
        
        comp_metrics = utils.evaluate_component_fidelity(
            true_comps, pred_comps, offset=s.fs.row_offset, level=LEVEL
        )
        
        results_asym.append({
            "trial": i,
            "asymmetric": asym,
            "trend_rmse": comp_metrics["trend"]["rmse"],
            "seasonal_rmse": comp_metrics["seasonal"]["rmse"],
            "generic_rmse": comp_metrics["generic"]["rmse"]
        })
print("Asymmetric prior ablation completed.")

In [ ]:
df_asym = pd.DataFrame(results_asym)
print("\n--- Asymmetric Prior Ablation Summary (Component Recovery RMSE) ---")
print(df_asym.groupby("asymmetric")[["trend_rmse", "seasonal_rmse", "generic_rmse"]].mean())

## 4. Sum-to-Zero Seasonal Centering

Without sum-to-zero seasonal centering, the seasonal block can absorb level/constant information that should be handled by the trend block. We compare `sum_to_zero=True` vs `sum_to_zero=False` on DGP 1, measuring in-sample component recovery RMSE.

In [ ]:
results_centering = []

print("Running Seasonal Centering ablation on DGP 1...")
for i in range(NUM_SIMULATIONS):
    t, y, true_comps = utils.generate_dgp1(n=200, seed=i)
    
    for center in [True, False]:
        cfg = make_config(
            periods=[(12, 3)],
            lags=(1,),
            trend="spline",
            errors="homo",
            sum_to_zero=center,
            num_gfr=10,
            num_burnin=120,
            num_mcmc=250,
            seed=i
        )
        
        model = BBEATSx(cfg).fit(y)
        s = model.sampler_
        mean, std = s.y_mean_, s.y_std_
        in_sample_draws = s.in_sample_components()
        
        pred_comps = {
            "trend": mean + std * in_sample_draws["trend"],
            "seasonal": std * in_sample_draws["seasonal"],
            "generic": std * in_sample_draws["generic"]
        }
        
        comp_metrics = utils.evaluate_component_fidelity(
            true_comps, pred_comps, offset=s.fs.row_offset, level=LEVEL
        )
        
        results_centering.append({
            "trial": i,
            "sum_to_zero": center,
            "trend_rmse": comp_metrics["trend"]["rmse"],
            "seasonal_rmse": comp_metrics["seasonal"]["rmse"]
        })
print("Seasonal Centering ablation completed.")

In [ ]:
df_center = pd.DataFrame(results_centering)
print("\n--- Seasonal Centering Ablation Summary (Component Recovery RMSE) ---")
print(df_center.groupby("sum_to_zero")[["trend_rmse", "seasonal_rmse"]].mean())